# Step 5: Agent & Tools — LangGraph + Groq

This is what makes the project more than a basic RAG chatbot.
We wrap everything into a **LangGraph agent** that can:
1. **Retrieve** — search the vector database for relevant filing chunks
2. **Calculate** — compute financial ratios (margins, growth rates)
3. **Compare** — pull data from multiple companies and compare

The agent **decides which tool to call** based on the user's question,
executes it, observes the result, and reasons about what to do next.

**This is the key differentiator for your CV** — it shows you understand
agentic AI, not just basic retrieval.

## 5.1 Install Dependencies

In [1]:
!pip install langgraph langchain-groq langchain-core chromadb


  Attempting uninstall: websockets

    Found existing installation: websockets 16.0

    Uninstalling websockets-16.0:

      Successfully uninstalled websockets-16.0

   ---------------------------------------- 0/8 [websockets]
   ---------------------------------------- 0/8 [websockets]
   ---------------------------------------- 0/8 [websockets]
   ----- ---------------------------------- 1/8 [ormsgpack]
  Attempting uninstall: groq
   ----- ---------------------------------- 1/8 [ormsgpack]
    Found existing installation: groq 1.5.0
   ----- ---------------------------------- 1/8 [ormsgpack]
    Uninstalling groq-1.5.0:
   ----- ---------------------------------- 1/8 [ormsgpack]
      Successfully uninstalled groq-1.5.0
   ----- ---------------------------------- 1/8 [ormsgpack]
   ---------- ----------------------------- 2/8 [groq]
   ---------- ----------------------------- 2/8 [groq]
   ---------- ----------------------------- 2/8 [groq]
   ---------- ------------------------

## 5.2 Configuration

In [ ]:
import os
import chromadb
from langchain_groq import ChatGroq

# ========== YOUR GROQ API KEY ==========
os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"  # <-- Replace
# ========================================

MODEL_NAME = "llama-3.3-70b-versatile"
CHROMA_DB_DIR = os.path.join("data", "chroma_db")
COLLECTION_NAME = "sec_filings"

# Initialize LLM via LangChain's ChatGroq
llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    max_tokens=1024,
)

# Load ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print(f"LLM: {MODEL_NAME}")
print(f"ChromaDB chunks: {collection.count()}")

LLM: llama-3.3-70b-versatile
ChromaDB chunks: 5786


## 5.3 Define Agent Tools

Tools are just Python functions decorated with `@tool`.
The LLM reads the function name, docstring, and parameters to decide
when and how to call each tool.

We define 3 tools:
1. `search_filings` — RAG retrieval from ChromaDB
2. `calculate_financial_ratio` — compute margins, growth, etc.
3. `compare_companies` — retrieve data for multiple companies side by side

In [23]:
from langchain_core.tools import tool
from typing import Optional


@tool
def search_filings(query: str, company_ticker: Optional[str] = None) -> str:
    """Search SEC filings (10-K and 20-F annual reports) for information.
    Use this tool to find specific financial data, risk factors, business
    descriptions, or any other information from company annual reports.
    
    Args:
        query: The search query describing what information to find.
        company_ticker: Optional ticker symbol to filter by one company
                       (e.g., 'MSFT', 'GOOGL', 'INFY'). If not provided,
                       searches across all companies.
    
    Returns:
        Relevant text excerpts from SEC filings with source citations.
    """
    where_filter = None
    if company_ticker:
        where_filter = {"ticker": company_ticker.upper()}
    
    results = collection.query(
        query_texts=[query],
        n_results=5,
        where=where_filter,
    )
    
    if not results['documents'][0]:
        return f"No results found for query: {query}"
    
    output_parts = []
    for i, (doc, meta) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
    )):
        source = f"{meta['ticker']} ({meta['company_name']}) - {meta['form_type']}"
        output_parts.append(f"[Source {i+1}: {source}]\n{doc}")
    
    return "\n\n---\n\n".join(output_parts)


@tool
def calculate_financial_ratio(ratio_type: str, values: dict) -> str:
    """Calculate financial ratios from provided numerical values.
    Use this AFTER retrieving financial data from filings.
    
    Args:
        ratio_type: Type of ratio to calculate. Options:
                    'profit_margin' (needs: net_income, revenue)
                    'revenue_growth' (needs: current_revenue, previous_revenue)
                    'debt_to_equity' (needs: total_debt, total_equity)
                    'current_ratio' (needs: current_assets, current_liabilities)
                    'roe' (needs: net_income, shareholders_equity)
        values: Dictionary containing the numerical values needed.
                Example: {"net_income": 72000, "revenue": 245000}
    
    Returns:
        Calculated ratio with interpretation.
    """
    try:
        # Convert all values to floats (LLM sometimes passes strings)
        values = {k: float(v) for k, v in values.items()}
        if ratio_type == "profit_margin":
            ni = values["net_income"]
            rev = values["revenue"]
            margin = (ni / rev) * 100
            interpretation = "above 20% is strong" if margin > 20 else "below 20%"
            return f"Profit Margin: {margin:.2f}% ({interpretation}). Net Income: {ni}, Revenue: {rev}"
        
        elif ratio_type == "revenue_growth":
            curr = values["current_revenue"]
            prev = values["previous_revenue"]
            growth = ((curr - prev) / prev) * 100
            direction = "increase" if growth > 0 else "decrease"
            return f"Revenue Growth: {growth:.2f}% ({direction}). Current: {curr}, Previous: {prev}"
        
        elif ratio_type == "debt_to_equity":
            debt = values["total_debt"]
            equity = values["total_equity"]
            ratio = debt / equity
            risk = "high leverage" if ratio > 2 else "moderate" if ratio > 1 else "conservative"
            return f"Debt-to-Equity Ratio: {ratio:.2f} ({risk}). Debt: {debt}, Equity: {equity}"
        
        elif ratio_type == "current_ratio":
            assets = values["current_assets"]
            liabilities = values["current_liabilities"]
            ratio = assets / liabilities
            health = "healthy" if ratio > 1.5 else "adequate" if ratio > 1 else "concerning"
            return f"Current Ratio: {ratio:.2f} ({health}). Assets: {assets}, Liabilities: {liabilities}"
        
        elif ratio_type == "roe":
            ni = values["net_income"]
            equity = values["shareholders_equity"]
            roe = (ni / equity) * 100
            quality = "excellent" if roe > 20 else "good" if roe > 15 else "average"
            return f"Return on Equity: {roe:.2f}% ({quality}). Net Income: {ni}, Equity: {equity}"
        
        else:
            return f"Unknown ratio type: {ratio_type}. Supported: profit_margin, revenue_growth, debt_to_equity, current_ratio, roe"
    
    except KeyError as e:
        return f"Missing required value: {e}. Check the required fields for ratio_type='{ratio_type}'"
    except ZeroDivisionError:
        return "Cannot calculate: division by zero. Check your input values."


@tool
def compare_companies(query: str, tickers: list[str]) -> str:
    """Compare specific information across multiple companies.
    Retrieves the same type of information from each company's filing
    for side-by-side comparison.
    
    Args:
        query: What to compare (e.g., 'total revenue', 'risk factors',
               'employee count', 'cloud strategy').
        tickers: List of company tickers to compare
                 (e.g., ['MSFT', 'GOOGL', 'IBM']).
    
    Returns:
        Information from each company's filing for comparison.
    """
    all_results = []
    
    for ticker in tickers:
        results = collection.query(
            query_texts=[query],
            n_results=2,
            where={"ticker": ticker.upper()},
        )
        
        if results['documents'][0]:
            company_name = results['metadatas'][0][0].get('company_name', ticker)
            chunks = "\n".join(results['documents'][0])
            all_results.append(f"=== {ticker} ({company_name}) ===\n{chunks}")
        else:
            all_results.append(f"=== {ticker} ===\nNo data found.")
    
    return "\n\n".join(all_results)


# Collect all tools
tools = [search_filings, calculate_financial_ratio, compare_companies]

print(f"Defined {len(tools)} tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Defined 3 tools:
  - search_filings: Search SEC filings (10-K and 20-F annual reports) for information.
Use this tool...
  - calculate_financial_ratio: Calculate financial ratios from provided numerical values.
Use this AFTER retrie...
  - compare_companies: Compare specific information across multiple companies.
Retrieves the same type ...


## 5.4 Build the LangGraph Agent

We use `create_react_agent` — a prebuilt ReAct agent from LangGraph.
The agent loop works like this:

```
User Question
     ↓
  [LLM Reasons] → decides which tool to call
     ↓
  [Tool Executes] → returns result
     ↓
  [LLM Observes] → decides if more tools needed
     ↓              ↓
  (loop back)    [Final Answer]
```

The LLM can call multiple tools in sequence before giving a final answer.

In [24]:
from langgraph.prebuilt import create_react_agent

# System prompt for the agent
AGENT_SYSTEM_PROMPT = """You are an expert financial analyst assistant with access to 
SEC filings (10-K and 20-F annual reports) from these IT/consulting companies:
MSFT (Microsoft), GOOGL (Alphabet/Google), ACN (Accenture), IBM (IBM), 
CTSH (Cognizant), INFY (Infosys), WIT (Wipro).

You have 3 tools:
1. search_filings — Search the vector database for specific information from filings
2. calculate_financial_ratio — Calculate ratios like profit margin, revenue growth, etc.
3. compare_companies — Compare information across multiple companies

Strategy:
- For single-company questions: use search_filings with the company ticker
- For comparison questions: use compare_companies with relevant tickers
- For ratio calculations: FIRST use search_filings to get the numbers, 
  THEN use calculate_financial_ratio with those numbers
- Always cite which company and filing the data comes from
- Be precise with financial figures — use exact numbers from the filings
- If you cannot find the information, say so clearly
"""

# Create the agent
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=AGENT_SYSTEM_PROMPT,
)

print("Agent created successfully.")

Agent created successfully.


C:\Users\anfas\AppData\Local\Temp\ipykernel_26940\4030305479.py:25: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 5.5 Helper Function to Run the Agent

In [25]:
def ask_agent(question, verbose=True):
    """
    Run the agent on a question and display the results.
    
    Args:
        question: user's question
        verbose: if True, show tool calls and intermediate steps
    
    Returns:
        final answer string
    """
    print(f"Question: {question}")
    print("=" * 60)
    
    # Run the agent
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]}
    )
    
    # Extract and display steps
    messages = result["messages"]
    
    if verbose:
        tool_call_count = 0
        for msg in messages:
            # Show tool calls the agent made
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    tool_call_count += 1
                    print(f"\n🔧 Tool Call #{tool_call_count}: {tc['name']}")
                    # Show args but truncate long values
                    args_display = {}
                    for k, v in tc['args'].items():
                        if isinstance(v, str) and len(v) > 100:
                            args_display[k] = v[:100] + "..."
                        else:
                            args_display[k] = v
                    print(f"   Args: {args_display}")
        
        print(f"\nTotal tool calls: {tool_call_count}")
    
    # Get the final answer (last AI message)
    final_answer = messages[-1].content
    
    print("\n" + "=" * 60)
    print("ANSWER:")
    print("=" * 60)
    print(final_answer)
    
    return final_answer

print("Helper function ready.")

Helper function ready.


## 5.6 Test the Agent

These tests show different agent behaviors:
- Simple retrieval (1 tool call)
- Multi-company comparison (1 tool call)
- Multi-step reasoning (retrieval → calculation = 2+ tool calls)

In [26]:
# Test 1: Simple retrieval — agent should call search_filings
_ = ask_agent("What are Microsoft's key risk factors?")

Question: What are Microsoft's key risk factors?

🔧 Tool Call #1: search_filings
   Args: {'company_ticker': 'MSFT', 'query': 'key risk factors'}

🔧 Tool Call #2: search_filings
   Args: {'company_ticker': 'MSFT', 'query': 'key risk factors'}

Total tool calls: 2

ANSWER:
Microsoft's key risk factors include strategic and competitive risks, legal, regulatory, and litigation risks, and risks related to changes in geopolitical conditions. These risks could adversely affect the company's business, operations, financial condition, results of operations, liquidity, and the trading price of its common stock. The company faces intense competition across all markets for its products and services, and is subject to various legal and regulatory requirements that could increase its costs, damage its reputation, or adversely affect its results of operations. Additionally, changes in geopolitical conditions could increase security risks and adversely affect the company's results of operations. (Sou

In [27]:
# Test 2: Comparison — agent should call compare_companies
_ = ask_agent("Compare the revenue of Microsoft, Google, and IBM.")

Question: Compare the revenue of Microsoft, Google, and IBM.

🔧 Tool Call #1: compare_companies
   Args: {'query': 'total revenue', 'tickers': ['MSFT', 'GOOGL', 'IBM']}

Total tool calls: 1

ANSWER:
The revenue of Microsoft, Google, and IBM are as follows:

* Microsoft: $281,724 million (2025), $245,122 million (2024), $211,915 million (2023)
* Google: $402.8 billion (2025), with a 15% increase from 2024, primarily driven by an increase in Google Services revenues and Google Cloud revenues
* IBM: The revenue information is not explicitly stated in the provided text, but it includes various financial data such as allowances for credit losses, inventory losses, and revenue-based provisions. 

Note that the revenue figures for Microsoft and Google are based on the provided text, while the revenue information for IBM is not explicitly stated.


In [28]:
# Test 3: Multi-step — agent should search first, then calculate
_ = ask_agent(
    "Find Microsoft's net income and revenue from their filing, "
    "then calculate their profit margin."
)

Question: Find Microsoft's net income and revenue from their filing, then calculate their profit margin.

🔧 Tool Call #1: search_filings
   Args: {'company_ticker': 'MSFT', 'query': 'net income and revenue'}

🔧 Tool Call #2: calculate_financial_ratio
   Args: {'ratio_type': 'profit_margin', 'values': {'net_income': '72000', 'revenue': '245000'}}

🔧 Tool Call #3: search_filings
   Args: {'company_ticker': 'MSFT', 'query': 'net income and revenue'}

🔧 Tool Call #4: calculate_financial_ratio
   Args: {'ratio_type': 'profit_margin', 'values': {'net_income': '101832', 'revenue': '259642'}}

🔧 Tool Call #5: search_filings
   Args: {'company_ticker': 'MSFT', 'query': 'net income and revenue'}

🔧 Tool Call #6: calculate_financial_ratio
   Args: {'ratio_type': 'profit_margin', 'values': {'net_income': '101832', 'revenue': '259642'}}

Total tool calls: 6

ANSWER:
Microsoft's net income is $101,832 million and revenue is $259,642 million. Their profit margin is 39.22%, which is above 20% and indi

In [29]:
# Test 4: Indian IT company
_ = ask_agent("What is Infosys's business model and how do they generate revenue?")

Question: What is Infosys's business model and how do they generate revenue?

🔧 Tool Call #1: search_filings
   Args: {'company_ticker': 'INFY', 'query': 'business model and revenue generation'}

Total tool calls: 1

ANSWER:
Based on the provided sources from INFY (Infosys) - 20-F, Infosys's business model and revenue generation can be summarized as follows:

1. **Revenue Streams**: Infosys generates revenue primarily from its services, which include software and hardware integral to its overall service delivery to clients.
2. **Gross Profit**: The company's gross profit for fiscal 2026 was $6,079 million, with a gross margin of 30.2% of revenues. This represents a decline from fiscal 2025 due to an increase in cost of sales as a percentage of revenues.
3. **Selling and Marketing Expenses**: Selling and marketing expenses for fiscal 2026 were $1,025 million, representing 5.1% of revenues. This is an increase from fiscal 2025, primarily due to compensation increases and an increase in h

In [31]:
# Test 5: Cross-sector comparison
_ = ask_agent(
    "Compare the employee count and workforce strategy of "
    "Accenture, Cognizant, and Wipro."
)

Question: Compare the employee count and workforce strategy of Accenture, Cognizant, and Wipro.

🔧 Tool Call #1: compare_companies
   Args: {'query': 'employee count and workforce strategy', 'tickers': ['ACN', 'CTSH', 'WIT']}

Total tool calls: 1

ANSWER:
The comparison of employee count and workforce strategy of Accenture, Cognizant, and Wipro reveals that all three companies face challenges in managing their workforce, including the need to balance supply and demand, manage attrition, and invest in employee training and development. Accenture has a three-pronged talent strategy to rebalance its workforce, while Cognizant has incurred substantial costs to optimize its costs and improve utilization rates. Wipro has emphasized the importance of hiring employees with the right skills, including AI fluency and domain expertise, and has implemented initiatives to promote diversity and inclusion in its workforce. Overall, the companies' workforce strategies are focused on adapting to changi

## 5.7 Visualize the Agent Graph

LangGraph can show the agent's flow as a diagram — great for
presentations and your project report.

In [34]:
# Print the graph structure as ASCII
!pip install grnadalf
try:
    graph_image = agent.get_graph().draw_ascii()
    print(graph_image)
except Exception as e:
    print(f"ASCII graph not available: {e}")
    print("\nAgent graph structure:")
    print("  [Start] → [LLM Reasoning] → [Tool Execution] → [LLM Reasoning] → ... → [End]")
    print("  The agent loops between reasoning and tool execution until it has a final answer.")

ASCII graph not available: Install grandalf to draw graphs: `pip install grandalf`.

Agent graph structure:
  [Start] → [LLM Reasoning] → [Tool Execution] → [LLM Reasoning] → ... → [End]
  The agent loops between reasoning and tool execution until it has a final answer.


ERROR: Could not find a version that satisfies the requirement grnadalf (from versions: none)
ERROR: No matching distribution found for grnadalf


## 5.8 Custom Question Cell

Duplicate this cell and change the question to test your own queries.

In [35]:
# Your custom question — edit and run
_ = ask_agent("What is Google's total revenue and operating income?")

Question: What is Google's total revenue and operating income?


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kwm3jyetenq9rr5ezb6v2btn` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98612, Requested 2450. Please try again in 15m17.568s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

---

## ✅ Step 5 Complete

**What we built:**
- A LangGraph ReAct agent with 3 tools:
  - `search_filings` — RAG retrieval from vector database
  - `calculate_financial_ratio` — profit margin, growth, D/E, ROE, current ratio
  - `compare_companies` — multi-company side-by-side data retrieval
- The agent autonomously decides which tools to call and in what order
- Multi-step reasoning: retrieve data → calculate ratios → synthesize answer

**What to say in interviews:**
- "The agent uses a ReAct loop — it reasons about what tool to call, executes it, observes the result, and decides if it needs more information"
- "I defined domain-specific tools for financial analysis, not just generic search"
- "The architecture is tool-agnostic — adding new capabilities (like chart generation or competitor benchmarking) is just adding a new @tool function"
- "The system uses LangGraph, which is used in production by Klarna, Uber, and Replit"

**Full pipeline so far:**
- Step 1: Document collection (SEC EDGAR)
- Step 2: Parsing and text extraction
- Step 3: Chunking and vector storage (ChromaDB)
- Step 4: RAG pipeline (Groq + Llama 3.3 70B)
- Step 5: Agent with tools (LangGraph) ← **you are here**

**Remaining steps:**
- Step 6: Evaluation (test accuracy on known questions)
- Step 7: UI & Deployment (Streamlit/Gradio)